<h1 align="center">Practical No. 04</h1>
<h2 align="center">Generating Kuzushiji (Classical Japanese Cursive) Characters<br>with a Deep Convolutional GAN</h2>

| | |
|---|---|
| **Name** | ____________________ |
| **Roll No. / PRN** | ____________________ |
| **Division / Batch** | ____________________ |
| **Subject** | Deep Learning |
| **Dataset** | Kuzushiji-MNIST (KMNIST), 70,000 images, 28 x 28, grayscale, 10 classes |
| **Model** | DCGAN (Radford, Metz & Chintala, 2016) |
| **Framework** | TensorFlow / Keras |

---

## Problem statement
Implement and train a **Deep Convolutional Generative Adversarial Network (DCGAN)** on a standard image dataset,
generate synthetic images with the trained model, and evaluate performance by analysing the **Generator and
Discriminator loss curves** and the **quality of the generated images**.

## Contents
| # | Section | What it does |
|---|---|---|
| 0 | Theory in brief | GAN game, loss functions, DCGAN design rules |
| 1 | Environment and configuration | Imports, seeds, hyper-parameter table |
| 2 | Dataset study | Load KMNIST, class table, class balance, mean images, pixel statistics |
| 3 | Pre-processing | Scale to [-1, 1] |
| 4 | Network design | Discriminator, Generator, parameter counts, untrained output |
| 5 | Training engine | A `DCGAN` class with one-sided label smoothing and gradient tracking |
| 6 | Training | Per-iteration logging, snapshots, checkpoints, training GIF |
| 7 | Loss-curve analysis | Four diagnostic plots |
| 8 | Image generation | Final grid, real vs fake, progress, interpolation, truncation, class gallery |
| 9 | Quantitative evaluation | Discriminator test, classifier score, Frechet distance, diversity, memorisation check |
| 10 | Automatic results summary | Numbers collected into one table and a written verdict |
| 11 | Discussion, limitations, viva questions, references | |

## 0. Theory in brief

### 0.1 The GAN idea
A GAN trains **two neural networks against each other**:

* **Generator G** takes a random noise vector `z` (here 100 numbers drawn from a normal distribution) and turns it into an image `G(z)`.
  It is the *forger*.
* **Discriminator D** looks at an image and outputs the probability that it is a **real** training image rather than one made by G.
  It is the *detective*.

D is rewarded for catching fakes; G is rewarded for fooling D. As both improve, G's pictures become closer and closer to the real data distribution.

### 0.2 The minimax objective

$$\min_G \max_D \; V(D,G)=\mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$$

In code (binary cross-entropy, BCE) this becomes:

| Player | Loss that is minimised | Meaning |
|---|---|---|
| Discriminator | `BCE(D(real), 1) + BCE(D(G(z)), 0)` | say "real" for real images and "fake" for generated ones |
| Generator | `BCE(D(G(z)), 1)` | make D say "real" for generated images (the *non-saturating* trick, which gives stronger gradients early in training than minimising `log(1 - D(G(z)))`) |

**Theoretical equilibrium:** G reproduces the data distribution exactly and D can only guess, so `D(x) = 0.5` everywhere.
Then the discriminator loss is `2 ln 2 = 1.386` and the generator loss is `ln 2 = 0.693`. Real training only hovers around this region.

### 0.3 What makes it a *Deep Convolutional* GAN
The DCGAN paper gave a set of architecture rules that made GAN training stable. This notebook follows each of them:

| DCGAN guideline | How it is applied here |
|---|---|
| Replace pooling with **strided convolutions** (D) and **fractional-strided / transposed convolutions** (G) | D downsamples with `stride=2` convolutions, G upsamples with `Conv2DTranspose` |
| Use **batch normalisation** in G | after every G layer except the output |
| Remove fully-connected hidden layers | only one Dense layer in G (projection of `z`) and one in D (final sigmoid) |
| **ReLU** in G, **LeakyReLU** in D | LeakyReLU (slope 0.2) is used in both, which avoids dead units and is common in practice |
| **Tanh** at the G output | output range [-1, 1], real images are scaled to the same range |
| **Adam** with `lr = 0.0002`, `beta_1 = 0.5` | exactly these values |

### 0.4 Extra stabilisers used in this notebook
* **One-sided label smoothing:** real targets are `0.9` instead of `1.0`, so D never becomes over-confident.
* **Dropout (0.4)** before D's output layer.
* **Gradient-norm tracking** for both networks, to spot vanishing gradients early.

## 1. Environment and configuration
The first cell installs a package that `tensorflow_datasets` needs on newer Colab images (Python 3.13) and removes any
partially downloaded copy of the dataset. Run it once; it is harmless if run again.

In [ ]:
# Colab fix: tensorflow_datasets needs importlib_resources, which newer Colab images do not ship.
!pip install -q importlib_resources
!rm -rf /root/tensorflow_datasets/kmnist

In [ ]:
import os, time, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
from PIL import Image
from IPython.display import display, Image as IPyImage

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 2026
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------ hyper-parameters
CFG = dict(
    latent_dim   = 100,      # length of the noise vector z
    epochs       = 50,       # passes over the 60,000 training images
    batch_size   = 128,
    lr           = 2e-4,     # Adam learning rate (DCGAN paper)
    beta_1       = 0.5,      # Adam beta_1 (DCGAN paper)
    real_label   = 0.9,      # one-sided label smoothing (1.0 would mean no smoothing)
    dropout      = 0.4,
    leak         = 0.2,      # LeakyReLU negative slope
    ckpt_every   = 10,       # save generator weights every N epochs
)
OUT = 'kmnist_dcgan_outputs'
os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------------ plotting style
plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titleweight': 'bold', 'axes.titlesize': 12, 'font.size': 10,
})
COL = dict(D='#2563eb', G='#dc2626', real='#059669', fake='#d97706', grey='#6b7280')

print('Python      :', platform.python_version())
print('TensorFlow  :', tf.__version__)
print('Keras       :', keras.__version__)
print('GPU devices :', tf.config.list_physical_devices('GPU') or 'none (training will be slow)')
display(pd.DataFrame(CFG.items(), columns=['hyper-parameter', 'value']).set_index('hyper-parameter'))

## 2. Dataset study: Kuzushiji-MNIST

**Kuzushiji** (崩し字) is the cursive style of Japanese writing used for roughly a thousand years until the late 19th century.
KMNIST (Clanuwat et al., 2018) was built from scanned pre-modern books of the *Center for Open Data in the Humanities* and
has the same size and format as MNIST, but the characters are far more varied: the same Hiragana can be written in many different ways.

| Class id | Romaji | Hiragana | | Class id | Romaji | Hiragana |
|---|---|---|---|---|---|---|
| 0 | o | お | | 5 | ha | は |
| 1 | ki | き | | 6 | ma | ま |
| 2 | su | す | | 7 | ya | や |
| 3 | tsu | つ | | 8 | re | れ |
| 4 | na | な | | 9 | wo | を |

Facts: 60,000 training + 10,000 test images, 28 x 28 pixels, 1 channel, pixel values 0 to 255, 6,000 training images per class.
For a GAN the labels are **not used during training** (it is unsupervised); they are used only later, by the evaluation classifier.

In [ ]:
CLASS_NAMES = ['o', 'ki', 'su', 'tsu', 'na', 'ha', 'ma', 'ya', 're', 'wo']

def _load_with_tfds():
    import tensorflow_datasets as tfds
    tr = tfds.as_numpy(tfds.load('kmnist', split='train', batch_size=-1))
    te = tfds.as_numpy(tfds.load('kmnist', split='test',  batch_size=-1))
    return (tr['image'], tr['label']), (te['image'], te['label'])

def _load_from_url():
    # fallback: official CODH mirror (.npz files)
    base = 'http://codh.rois.ac.jp/kmnist/dataset/kmnist/'
    out = []
    for name in ['kmnist-train-imgs.npz', 'kmnist-train-labels.npz', 'kmnist-test-imgs.npz', 'kmnist-test-labels.npz']:
        path = keras.utils.get_file(name, base + name)
        out.append(np.load(path)['arr_0'])
    return (out[0][..., None], out[1]), (out[2][..., None], out[3])

try:
    (trainX, trainy), (testX, testy) = _load_with_tfds()
    print('Loaded KMNIST through tensorflow_datasets')
except Exception as err:
    print('tensorflow_datasets failed (%s: %s) -> using direct download' % (type(err).__name__, err))
    (trainX, trainy), (testX, testy) = _load_from_url()

print('train images :', trainX.shape, trainX.dtype)
print('train labels :', trainy.shape)
print('test images  :', testX.shape)
print('pixel range  : %d to %d' % (trainX.min(), trainX.max()))

In [ ]:
# ---- 2.1 one row per class: ten random real examples of every character
fig, axes = plt.subplots(10, 10, figsize=(8, 8.4))
for c in range(10):
    idx = rng.choice(np.where(trainy == c)[0], 10, replace=False)
    for j, k in enumerate(idx):
        ax = axes[c, j]
        ax.imshow(trainX[k].squeeze(), cmap='gray_r'); ax.axis('off')
    axes[c, 0].set_title(CLASS_NAMES[c], loc='left', fontsize=9, color=COL['G'])
plt.suptitle('Real KMNIST characters, one row per class (note the many handwriting styles)', y=0.93)
plt.show()

In [ ]:
# ---- 2.2 class balance + average image per class + pixel statistics
fig = plt.figure(figsize=(14, 4.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.6, 1])

ax = fig.add_subplot(gs[0])
counts = np.bincount(trainy, minlength=10)
ax.bar(CLASS_NAMES, counts, color=COL['D'])
ax.set_title('Training images per class'); ax.set_ylim(0, counts.max() * 1.15)
for i, v in enumerate(counts): ax.text(i, v + 80, str(v), ha='center', fontsize=8)

ax = fig.add_subplot(gs[1]); ax.grid(False)
means = np.stack([trainX[trainy == c].mean(axis=0).squeeze() for c in range(10)])
sheet = np.vstack([np.hstack(means[:5]), np.hstack(means[5:])])
ax.imshow(sheet, cmap='gray_r'); ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Average image of each class (row 1: o ki su tsu na, row 2: ha ma ya re wo)')

ax = fig.add_subplot(gs[2])
ax.hist(trainX.ravel(), bins=50, color=COL['real'], log=True)
ax.set_title('Pixel intensity histogram (log scale)'); ax.set_xlabel('pixel value')
plt.tight_layout(); plt.show()

print('Mean pixel value: %.1f | std: %.1f | fraction of pure-background pixels (0): %.1f%%'
      % (trainX.mean(), trainX.std(), (trainX == 0).mean() * 100))

**Reading the dataset plots:** the classes are perfectly balanced (6,000 each), so a generator that covers the data
should produce all ten characters in roughly equal numbers. Most pixels are background and ink strokes form a thin bright
skeleton, so the generator must learn *thin, connected strokes* rather than blobs. The pixel histogram is strongly
bimodal, which is why a `tanh` output (saturating near -1 and +1) suits this data.

## 3. Pre-processing
Pixels are converted to `float32` and linearly mapped from [0, 255] to **[-1, 1]**:
`x_scaled = (x - 127.5) / 127.5`. The generator's `tanh` output lives in the same range, so real and generated images are directly comparable.
A `to_unit()` helper converts back to [0, 1] for display.

In [ ]:
def scale_to_signed(images):
    return (images.astype('float32') - 127.5) / 127.5

def to_unit(images):
    return np.clip((images + 1.0) / 2.0, 0.0, 1.0)

X_train = scale_to_signed(trainX)
X_test  = scale_to_signed(testX)
print('X_train:', X_train.shape, '| min %.2f max %.2f mean %.3f' % (X_train.min(), X_train.max(), X_train.mean()))

# image helpers used all through the notebook
def mosaic(batch, rows, cols):
    # tile rows*cols images (in [-1,1]) into one picture with a thin white gap between them
    imgs = to_unit(batch[:rows * cols])[..., 0]
    imgs = np.pad(imgs, ((0, 0), (1, 1), (1, 1)), constant_values=1.0)
    h, w = imgs.shape[1:]
    return imgs.reshape(rows, cols, h, w).transpose(0, 2, 1, 3).reshape(rows * h, cols * w)

def show(batch, rows, cols, title='', fname=None, size=7):
    plt.figure(figsize=(size, size * rows / cols)); plt.grid(False)
    plt.imshow(mosaic(batch, rows, cols), cmap='gray_r'); plt.axis('off'); plt.title(title)
    if fname: plt.savefig(os.path.join(OUT, fname), dpi=150, bbox_inches='tight')
    plt.show()

## 4. Network design

### 4.1 Discriminator: 28 x 28 x 1 image -> probability "real"
```
Input 28x28x1
 -> Conv 64  (3x3, stride 1) + LeakyReLU          28x28x64
 -> Conv 128 (3x3, stride 2) + LeakyReLU          14x14x128   downsample
 -> Conv 128 (3x3, stride 2) + LeakyReLU          7x7x128     downsample
 -> Flatten -> Dropout(0.4) -> Dense(1, sigmoid)  one probability
```

In [ ]:
def build_discriminator(leak=CFG['leak'], drop=CFG['dropout']):
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, 3, padding='same'),                 layers.LeakyReLU(negative_slope=leak),
        layers.Conv2D(128, 3, strides=2, padding='same'),     layers.LeakyReLU(negative_slope=leak),
        layers.Conv2D(128, 3, strides=2, padding='same'),     layers.LeakyReLU(negative_slope=leak),
        layers.Flatten(),
        layers.Dropout(drop),
        layers.Dense(1, activation='sigmoid'),
    ], name='discriminator')

### 4.2 Generator: 100-dim noise -> 28 x 28 x 1 image
```
Input z (100)
 -> Dense 7*7*128 + BatchNorm + LeakyReLU -> reshape   7x7x128    project and reshape
 -> Conv2DTranspose 128 (4x4, stride 2) + BN + LeakyReLU 14x14x128 upsample
 -> Conv2DTranspose 64  (4x4, stride 2) + BN + LeakyReLU 28x28x64  upsample
 -> Conv 1 (7x7) + tanh                                  28x28x1    image in [-1, 1]
```
A `4x4` kernel with `stride 2` is used for the transposed convolutions because a kernel size divisible by the stride
avoids the *checkerboard artefacts* that a `3x3` kernel would create.

In [ ]:
def build_generator(latent_dim=CFG['latent_dim'], leak=CFG['leak']):
    return keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(7 * 7 * 128, use_bias=False), layers.BatchNormalization(), layers.LeakyReLU(negative_slope=leak),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(128, 4, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(negative_slope=leak),
        layers.Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(negative_slope=leak),
        layers.Conv2D(1, 7, padding='same', activation='tanh'),
    ], name='generator')

_g, _d = build_generator(), build_discriminator()
_g.summary(); _d.summary()
print('\nGenerator params     : %8d' % _g.count_params())
print('Discriminator params : %8d' % _d.count_params())

In [ ]:
# What does an UNTRAINED generator draw? (Expected: structureless noise. This is our "epoch 0" baseline.)
z0 = rng.standard_normal((49, CFG['latent_dim'])).astype('float32')
show(_g(z0, training=False).numpy(), 7, 7, 'Epoch 0: untrained generator output')
del _g, _d

## 5. Training engine
All logic lives in one `DCGAN` class. Each call of `train_step` performs **one simultaneous update** of both networks on a
fresh batch of real images and an equally large batch of generated images:

1. Draw noise `z` and generate fake images `G(z)`.
2. Pass real and fake batches through D.
3. **D loss** = BCE(real, **0.9**) + BCE(fake, 0). **G loss** = BCE(fake, 1).
4. Compute gradients of each loss with respect to *its own network only* (two `GradientTape`s) and apply them with separate Adam optimisers.

Besides the losses, the step returns diagnostics that explain *why* training behaves as it does:

| Diagnostic | What it tells you |
|---|---|
| `D(x)` - mean D output on real images | should stay around 0.5-0.9 |
| `D(G(z))` - mean D output on fake images | rises towards 0.5 as G improves; stuck near 0 means D is winning |
| accuracy on real / fake | 50% on fake = D is fooled |
| gradient norm of G / D | G's gradient collapsing to 0 = vanishing gradients (D too strong) |

In [ ]:
METRICS = ['d_loss', 'd_loss_real', 'd_loss_fake', 'g_loss', 'acc_real', 'acc_fake', 'd_x', 'd_gz', 'g_gnorm', 'd_gnorm']

class DCGAN:
    def __init__(self, cfg):
        self.cfg = cfg
        self.latent_dim = cfg['latent_dim']
        self.real_label = cfg['real_label']
        self.G = build_generator(cfg['latent_dim'], cfg['leak'])
        self.D = build_discriminator(cfg['leak'], cfg['dropout'])
        self.g_opt = keras.optimizers.Adam(cfg['lr'], beta_1=cfg['beta_1'])
        self.d_opt = keras.optimizers.Adam(cfg['lr'], beta_1=cfg['beta_1'])
        self.g_opt.build(self.G.trainable_variables)
        self.d_opt.build(self.D.trainable_variables)
        self.bce = keras.losses.BinaryCrossentropy()

    @tf.function
    def train_step(self, real):
        n = tf.shape(real)[0]
        z = tf.random.normal((n, self.latent_dim))
        with tf.GradientTape() as tape_d, tf.GradientTape() as tape_g:
            fake   = self.G(z, training=True)
            p_real = self.D(real, training=True)
            p_fake = self.D(fake, training=True)
            d_real = self.bce(tf.ones_like(p_real) * self.real_label, p_real)   # smoothed target
            d_fake = self.bce(tf.zeros_like(p_fake), p_fake)
            d_loss = d_real + d_fake
            g_loss = self.bce(tf.ones_like(p_fake), p_fake)                    # G wants D to say "real"
        grads_d = tape_d.gradient(d_loss, self.D.trainable_variables)
        grads_g = tape_g.gradient(g_loss, self.G.trainable_variables)
        self.d_opt.apply_gradients(zip(grads_d, self.D.trainable_variables))
        self.g_opt.apply_gradients(zip(grads_g, self.G.trainable_variables))
        return tf.stack([
            d_loss, d_real, d_fake, g_loss,
            tf.reduce_mean(tf.cast(p_real > 0.5, tf.float32)),
            tf.reduce_mean(tf.cast(p_fake < 0.5, tf.float32)),
            tf.reduce_mean(p_real), tf.reduce_mean(p_fake),
            tf.linalg.global_norm(grads_g), tf.linalg.global_norm(grads_d),
        ])

    def sample(self, n, scale=1.0, z=None):
        if z is None:
            z = (rng.standard_normal((n, self.latent_dim)) * scale).astype('float32')
        return self.G(z, training=False).numpy()

gan = DCGAN(CFG)
print('DCGAN ready | G params: %d | D params: %d' % (gan.G.count_params(), gan.D.count_params()))

> **Why not Keras `model.fit` with a frozen combined model (as in many tutorials)?**
> Freezing D by setting `trainable = False` after compiling is silently ignored in current Keras 3, which can make the
> generator update the discriminator as well. Two explicit gradient tapes guarantee that each network is updated by *its own* loss only.

## 6. Training
During training the notebook records

* all 10 diagnostics for **every iteration** (468 per epoch) and their **epoch averages**,
* a 7 x 7 grid at five chosen epochs (generated from the **same fixed noise** so progress is comparable),
* a small 4 x 4 grid after **every** epoch, which is later assembled into an animated GIF,
* generator weights every 10 epochs (checkpoints).

Expected speed: about 5-10 s per epoch on a Colab T4 GPU (roughly 5-8 minutes in total).

In [ ]:
train_ds = (tf.data.Dataset.from_tensor_slices(X_train)
            .shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
            .batch(CFG['batch_size'], drop_remainder=True)
            .prefetch(tf.data.AUTOTUNE))
steps_per_epoch = len(X_train) // CFG['batch_size']

it_log  = {m: [] for m in METRICS}      # every iteration
ep_log  = {m: [] for m in METRICS}      # mean per epoch
ep_time = []

fixed_z = rng.standard_normal((49, CFG['latent_dim'])).astype('float32')
E = CFG['epochs']
snap_epochs = sorted({1, max(2, E // 10), max(3, E // 4), E // 2, E})
snaps, gif_frames = {}, []

t_start = time.time()
for epoch in range(1, E + 1):
    t0 = time.time()
    buf = []
    for real_batch in train_ds:
        buf.append(gan.train_step(real_batch).numpy())
    buf = np.array(buf)                                   # (steps, 10)
    for k, m in enumerate(METRICS):
        it_log[m].extend(buf[:, k].tolist())
        ep_log[m].append(float(buf[:, k].mean()))
    ep_time.append(time.time() - t0)

    imgs = gan.G(fixed_z, training=False).numpy()
    tile = (mosaic(imgs, 4, 4) * 255).astype('uint8')
    gif_frames.append(Image.fromarray(255 - tile).resize((tile.shape[1] * 3, tile.shape[0] * 3), Image.NEAREST))
    if epoch in snap_epochs:
        snaps[epoch] = imgs
    if epoch % CFG['ckpt_every'] == 0 or epoch == E:
        gan.G.save_weights(os.path.join(OUT, 'generator_epoch_%03d.weights.h5' % epoch))

    print('Ep %3d/%d | D %.3f (real %.3f + fake %.3f) | G %.3f | D(x) %.2f D(G(z)) %.2f | acc real %3.0f%% fake %3.0f%% | %.1fs'
          % (epoch, E, ep_log['d_loss'][-1], ep_log['d_loss_real'][-1], ep_log['d_loss_fake'][-1], ep_log['g_loss'][-1],
             ep_log['d_x'][-1], ep_log['d_gz'][-1], ep_log['acc_real'][-1] * 100, ep_log['acc_fake'][-1] * 100, ep_time[-1]))

train_minutes = (time.time() - t_start) / 60
print('\nFinished %d epochs (%d iterations) in %.1f minutes' % (E, E * steps_per_epoch, train_minutes))

gan.G.save(os.path.join(OUT, 'generator_final.keras'))
gan.D.save(os.path.join(OUT, 'discriminator_final.keras'))
pd.DataFrame(ep_log).assign(epoch=np.arange(1, E + 1)).to_csv(os.path.join(OUT, 'epoch_history.csv'), index=False)

## 7. Loss-curve analysis

In [ ]:
def smooth(x, w=150):
    x = np.asarray(x, dtype='float64'); w = max(1, min(w, len(x)))
    return np.convolve(x, np.ones(w) / w, mode='valid')

ep_x = np.arange(1, E + 1)
fig, ax = plt.subplots(2, 2, figsize=(15, 9.5))

# (a) per-iteration losses
a = ax[0, 0]
a.plot(it_log['d_loss'], color=COL['D'], alpha=0.12); a.plot(it_log['g_loss'], color=COL['G'], alpha=0.12)
a.plot(np.arange(len(smooth(it_log['d_loss']))) + 75, smooth(it_log['d_loss']), color=COL['D'], lw=2, label='Discriminator loss')
a.plot(np.arange(len(smooth(it_log['g_loss']))) + 75, smooth(it_log['g_loss']), color=COL['G'], lw=2, label='Generator loss')
a.axhline(2 * np.log(2), color=COL['D'], ls=':', lw=1); a.axhline(np.log(2), color=COL['G'], ls=':', lw=1)
a.set_title('(a) Loss per iteration (faint) and 150-iteration moving average')
a.set_xlabel('iteration'); a.set_ylabel('binary cross-entropy'); a.legend()
a.text(0.99, 0.03, 'dotted = equilibrium values (D: 2 ln 2, G: ln 2)', transform=a.transAxes, ha='right', fontsize=8, color=COL['grey'])

# (b) per-epoch decomposition of D loss
a = ax[0, 1]
a.plot(ep_x, ep_log['d_loss'], color=COL['D'], lw=2, label='D total')
a.plot(ep_x, ep_log['d_loss_real'], color=COL['real'], ls='--', label='D on real images')
a.plot(ep_x, ep_log['d_loss_fake'], color=COL['fake'], ls='--', label='D on generated images')
a.plot(ep_x, ep_log['g_loss'], color=COL['G'], lw=2, label='G loss')
a.set_title('(b) Epoch-mean losses, with D split into real and fake parts')
a.set_xlabel('epoch'); a.set_ylabel('binary cross-entropy'); a.legend(ncol=2)

# (c) D outputs and accuracy
a = ax[1, 0]
a.plot(ep_x, ep_log['d_x'], color=COL['real'], lw=2, label='D(x), mean output on real')
a.plot(ep_x, ep_log['d_gz'], color=COL['fake'], lw=2, label='D(G(z)), mean output on fake')
a.axhline(0.5, color=COL['grey'], ls=':')
a.plot(ep_x, ep_log['acc_real'], color=COL['real'], ls='--', alpha=0.7, label='accuracy on real')
a.plot(ep_x, ep_log['acc_fake'], color=COL['fake'], ls='--', alpha=0.7, label='accuracy on fake')
a.set_ylim(0, 1.02); a.set_title('(c) Discriminator confidence and accuracy (0.5 = cannot tell)')
a.set_xlabel('epoch'); a.legend(ncol=2, fontsize=8)

# (d) gradient norms
a = ax[1, 1]
a.plot(ep_x, ep_log['g_gnorm'], color=COL['G'], lw=2, label='generator gradient norm')
a.plot(ep_x, ep_log['d_gnorm'], color=COL['D'], lw=2, label='discriminator gradient norm')
a.set_title('(d) Gradient norms (collapse to ~0 would mean vanishing gradients)')
a.set_xlabel('epoch'); a.set_ylabel('global L2 norm'); a.legend()

plt.tight_layout(); plt.savefig(os.path.join(OUT, 'loss_curves.png'), dpi=150); plt.show()

In [ ]:
# numeric summary of the curves: start vs middle vs end
def window(m, lo, hi): return float(np.mean(ep_log[m][lo:hi]))
n = max(3, E // 5)
rows = []
for label, lo, hi in [('first %d epochs' % n, 0, n), ('middle', E // 2 - n // 2, E // 2 + n // 2 + 1), ('last %d epochs' % n, E - n, E)]:
    rows.append([label] + [window(m, lo, hi) for m in ['d_loss', 'g_loss', 'd_x', 'd_gz', 'acc_real', 'acc_fake']])
curve_table = pd.DataFrame(rows, columns=['phase', 'D loss', 'G loss', 'D(x)', 'D(G(z))', 'acc real', 'acc fake']).set_index('phase')
display(curve_table.round(3))

# how noisy is training? (std of the smoothed-out iteration losses in the last quarter)
q = len(it_log['g_loss']) * 3 // 4
print('Std of per-iteration G loss, last quarter: %.3f | D loss: %.3f' % (np.std(it_log['g_loss'][q:]), np.std(it_log['d_loss'][q:])))

### How to interpret these plots

| What you see | What it means |
|---|---|
| Both losses **oscillate** in a band instead of falling smoothly | normal: each network's task keeps changing because the opponent improves |
| **G loss rises** over time while D loss falls | D is slowly winning; G sees weaker feedback. A gentle rise is common and is *not* by itself a failure |
| D loss near **0** and G loss shooting up (> 3-4) | D has taken over, G is receiving vanishing gradients -> training failed |
| G loss near 0 and images all look alike | **mode collapse**: G found one image that always fools D |
| D(G(z)) climbing from about 0 toward 0.3-0.5 | G is learning to fool D (healthy) |
| Both accuracies around 50-80% | healthy balance; 100% / 100% would mean D wins outright |

Note on label smoothing: because real targets are 0.9, D's loss on real images cannot go to 0 (its minimum is about 0.325), which keeps D
from becoming over-confident. The equilibrium value of the *total* D loss is still `2 ln 2`.

## 8. Image generation with the trained generator

In [ ]:
# 8.1 one hundred brand-new characters from fresh random noise
final_fake = gan.sample(100)
show(final_fake, 10, 10, '100 synthetic KMNIST characters (fresh noise, trained generator)', 'final_generated_100.png', size=8)

In [ ]:
# 8.2 real test images and generated images side by side
fig, ax = plt.subplots(1, 2, figsize=(13, 6.6))
ax[0].imshow(mosaic(X_test[rng.choice(len(X_test), 64, replace=False)], 8, 8), cmap='gray_r'); ax[0].set_title('REAL (held-out test set)')
ax[1].imshow(mosaic(gan.sample(64), 8, 8), cmap='gray_r'); ax[1].set_title('GENERATED by the DCGAN')
for a in ax: a.axis('off'); a.grid(False)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'real_vs_generated.png'), dpi=150); plt.show()

In [ ]:
# 8.3 how the same 25 noise vectors evolve during training
fig, axes = plt.subplots(1, len(snap_epochs), figsize=(3.3 * len(snap_epochs), 3.8))
for a, e in zip(np.atleast_1d(axes), snap_epochs):
    a.imshow(mosaic(snaps[e], 5, 5), cmap='gray_r'); a.set_title('epoch %d' % e); a.axis('off'); a.grid(False)
plt.suptitle('Training progress (identical noise vectors at every stage)', y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'training_progress.png'), dpi=150, bbox_inches='tight'); plt.show()

# animated version: one frame per epoch
gif_path = os.path.join(OUT, 'training_progress.gif')
gif_frames[0].save(gif_path, save_all=True, append_images=gif_frames[1:], duration=120, loop=0)
display(IPyImage(filename=gif_path))

In [ ]:
# 8.4 latent-space interpolation: every row morphs smoothly from one noise vector to another
rows_i, steps_i = 6, 10
za = rng.standard_normal((rows_i, CFG['latent_dim'])).astype('float32')
zb = rng.standard_normal((rows_i, CFG['latent_dim'])).astype('float32')
t = np.linspace(0, 1, steps_i).reshape(1, steps_i, 1).astype('float32')
z_path = ((1 - t) * za[:, None, :] + t * zb[:, None, :]).reshape(-1, CFG['latent_dim'])
show(gan.sample(len(z_path), z=z_path), rows_i, steps_i, 'Latent interpolation: smooth morphing = a well-structured latent space', 'latent_interpolation.png', size=12)

In [ ]:
# 8.5 truncation: shrinking the noise makes samples safer / more typical, enlarging it makes them wilder
scales = [0.3, 0.7, 1.0, 1.5, 2.5]
fig, axes = plt.subplots(1, len(scales), figsize=(3.2 * len(scales), 3.6))
for a, s in zip(axes, scales):
    a.imshow(mosaic(gan.sample(25, scale=s), 5, 5), cmap='gray_r'); a.set_title('noise std = %.1f' % s); a.axis('off'); a.grid(False)
plt.suptitle('Effect of the noise scale on the generated images (training used std = 1.0)', y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'truncation.png'), dpi=150, bbox_inches='tight'); plt.show()

## 9. Quantitative evaluation of image quality
Judging by eye is subjective, so five numerical checks are added.

| # | Check | Question it answers | Good result |
|---|---|---|---|
| A | Discriminator on unseen data | does D still separate real from fake? | accuracy on fakes not 100% |
| B | Classifier confidence and Inception-style score | are generated images clear *and* diverse? | close to the real-data values |
| C | Class histogram | does G cover all ten characters? | bars similar to the real bars |
| D | **Frechet distance** (FID-style) on classifier features | how far is the generated feature distribution from the real one? | small, close to the real-vs-real baseline |
| E | Diversity and memorisation | is G repeating itself, or copying training images? | diversity near real; nearest training image is *not* identical |

The classifier used for B, C and D is a small CNN trained on **real** KMNIST first (about 93-95% test accuracy).
Frechet distance is computed on its 128-dimensional hidden layer, which follows the idea of FID (Heusel et al., 2017) but with a
KMNIST-specific network instead of the ImageNet Inception network, so values are **not comparable with published FID numbers**.

In [ ]:
# ---------- A. discriminator on unseen real test images vs fresh fakes
fake_pool = gan.sample(10000)
p_real_test = gan.D(X_test, training=False).numpy().ravel()
p_fake_pool = gan.D(fake_pool, training=False).numpy().ravel()
acc_real_test = float((p_real_test > 0.5).mean())
acc_fake_pool = float((p_fake_pool < 0.5).mean())
print('A. Discriminator on 10,000 real TEST images : %.1f%% judged real, mean output %.3f' % (acc_real_test * 100, p_real_test.mean()))
print('   Discriminator on 10,000 generated images : %.1f%% judged fake, mean output %.3f' % (acc_fake_pool * 100, p_fake_pool.mean()))

plt.figure(figsize=(8, 3.6))
plt.hist(p_real_test, bins=40, alpha=0.65, color=COL['real'], label='real test images')
plt.hist(p_fake_pool, bins=40, alpha=0.65, color=COL['fake'], label='generated images')
plt.axvline(0.5, color=COL['grey'], ls=':'); plt.xlabel('D output (probability "real")'); plt.ylabel('count')
plt.title('Discriminator output distribution (heavy overlap = G is fooling D)'); plt.legend()
plt.savefig(os.path.join(OUT, 'discriminator_histogram.png'), dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ---------- reference classifier trained on REAL data only
inp = keras.Input((28, 28, 1))
h = layers.Conv2D(32, 3, activation='relu')(inp); h = layers.MaxPooling2D()(h)
h = layers.Conv2D(64, 3, activation='relu')(h);   h = layers.MaxPooling2D()(h)
h = layers.Flatten()(h)
h = layers.Dense(128, activation='relu', name='feat')(h)
h = layers.Dropout(0.3)(h)
out = layers.Dense(10, activation='softmax')(h)
clf = keras.Model(inp, out, name='reference_classifier')
clf.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
clf.fit(X_train, trainy, epochs=5, batch_size=128, validation_data=(X_test, testy), verbose=2)
feat_net = keras.Model(inp, clf.get_layer('feat').output)
clf_acc = clf.evaluate(X_test, testy, verbose=0)[1]
print('\nReference classifier test accuracy: %.2f%%' % (clf_acc * 100))

In [ ]:
# ---------- B + C. confidence, Inception-style score, class coverage
def inception_style_score(p, eps=1e-12):
    p_y = p.mean(axis=0, keepdims=True)
    return float(np.exp((p * (np.log(p + eps) - np.log(p_y + eps))).sum(axis=1).mean()))

p_real = clf.predict(X_test,     batch_size=512, verbose=0)
p_fake = clf.predict(fake_pool,  batch_size=512, verbose=0)

conf_real, conf_fake = float(p_real.max(1).mean()), float(p_fake.max(1).mean())
is_real,   is_fake   = inception_style_score(p_real), inception_style_score(p_fake)
share_real = np.bincount(p_real.argmax(1), minlength=10) / len(p_real) * 100
share_fake = np.bincount(p_fake.argmax(1), minlength=10) / len(p_fake) * 100

print('B. mean classifier confidence  real %.3f | generated %.3f' % (conf_real, conf_fake))
print('   Inception-style score (max 10)  real %.2f | generated %.2f' % (is_real, is_fake))

x = np.arange(10)
plt.figure(figsize=(10, 3.8))
plt.bar(x - 0.2, share_real, 0.4, color=COL['real'], label='real test images')
plt.bar(x + 0.2, share_fake, 0.4, color=COL['fake'], label='generated images')
plt.xticks(x, CLASS_NAMES); plt.ylabel('% of images'); plt.xlabel('class assigned by the reference classifier')
plt.title('C. Class coverage: matching bars = every character is generated'); plt.legend()
plt.savefig(os.path.join(OUT, 'class_distribution.png'), dpi=150, bbox_inches='tight'); plt.show()
classes_covered = int((share_fake >= 3).sum())
print('Classes that make up at least 3%% of the generated images: %d / 10' % classes_covered)

In [ ]:
# ---------- D. Frechet distance on the classifier's hidden features
def frechet_distance(f1, f2):
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    root = linalg.sqrtm(s1.dot(s2))
    root = root[0] if isinstance(root, tuple) else root
    if np.iscomplexobj(root): root = root.real
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1) + np.trace(s2) - 2.0 * np.trace(root))

sel = rng.choice(len(X_train), 10000, replace=False)
f_train = feat_net.predict(X_train[sel],  batch_size=512, verbose=0)
f_test  = feat_net.predict(X_test,        batch_size=512, verbose=0)
f_fake  = feat_net.predict(fake_pool,     batch_size=512, verbose=0)
noise_imgs = rng.uniform(-1, 1, (10000, 28, 28, 1)).astype('float32')
f_noise = feat_net.predict(noise_imgs, batch_size=512, verbose=0)

fd_baseline = frechet_distance(f_train, f_test)    # real vs real: the best a perfect generator could reach
fd_gan      = frechet_distance(f_train, f_fake)
fd_noise    = frechet_distance(f_train, f_noise)   # random pixel noise: a worst-case reference
print('D. Frechet distance to real training features')
print('   real test images (lower bound)  : %8.2f' % fd_baseline)
print('   DCGAN generated images          : %8.2f' % fd_gan)
print('   uniform random noise (reference): %8.2f' % fd_noise)

In [ ]:
# ---------- E. diversity and memorisation check
def pairwise_mean_distance(a):
    a = a.reshape(len(a), -1)
    sq = (a ** 2).sum(1)
    d2 = np.clip(sq[:, None] - 2 * a @ a.T + sq[None, :], 0, None)
    return float(np.sqrt(d2).sum() / (len(a) * (len(a) - 1)))

sub_real = X_test[rng.choice(len(X_test), 1000, replace=False)]
sub_fake = fake_pool[:1000]
div_real, div_fake = pairwise_mean_distance(sub_real), pairwise_mean_distance(sub_fake)
print('E. Mean pairwise pixel distance between images (higher = more diverse)')
print('   real %.2f | generated %.2f | ratio %.2f' % (div_real, div_fake, div_fake / div_real))

# nearest real training image for 10 generated samples (is the generator copying?)
pool = X_train[rng.choice(len(X_train), 20000, replace=False)].reshape(20000, -1)
g = fake_pool[:10].reshape(10, -1)
d2 = (g ** 2).sum(1)[:, None] - 2 * g @ pool.T + (pool ** 2).sum(1)[None, :]
nn_idx = d2.argmin(1)
nn_dist = np.sqrt(np.clip(d2.min(1), 0, None))

fig, axes = plt.subplots(2, 10, figsize=(14, 3.4))
for j in range(10):
    axes[0, j].imshow(to_unit(fake_pool[j, ..., 0]), cmap='gray_r'); axes[0, j].axis('off'); axes[0, j].grid(False)
    axes[1, j].imshow(to_unit(pool[nn_idx[j]].reshape(28, 28)), cmap='gray_r'); axes[1, j].axis('off'); axes[1, j].grid(False)
    axes[1, j].set_title('d=%.1f' % nn_dist[j], fontsize=8)
axes[0, 0].set_title('generated', loc='left', fontsize=9); axes[1, 0].text(-0.05, 1.18, 'closest real training image', transform=axes[1, 0].transAxes, fontsize=9)
plt.suptitle('Memorisation check: similar style but not identical copies = the GAN generalises', y=1.04)
plt.savefig(os.path.join(OUT, 'nearest_neighbour.png'), dpi=150, bbox_inches='tight'); plt.show()

# distance to nearest neighbour for a real test image, as a yardstick
t_img = X_test[:10].reshape(10, -1)
d2t = (t_img ** 2).sum(1)[:, None] - 2 * t_img @ pool.T + (pool ** 2).sum(1)[None, :]
print('Mean nearest-training distance: generated %.2f | real test images %.2f' % (nn_dist.mean(), np.sqrt(np.clip(d2t.min(1), 0, None)).mean()))

In [ ]:
# ---------- gallery: the 10 most confidently recognised generated characters of every class
big_fake = gan.sample(6000)
p_big = clf.predict(big_fake, batch_size=512, verbose=0)
fig, axes = plt.subplots(10, 10, figsize=(8, 8.4))
for c in range(10):
    order = np.argsort(-p_big[:, c])[:10]
    for j, k in enumerate(order):
        axes[c, j].imshow(to_unit(big_fake[k, ..., 0]), cmap='gray_r'); axes[c, j].axis('off'); axes[c, j].grid(False)
    axes[c, 0].set_title(CLASS_NAMES[c], loc='left', fontsize=9, color=COL['G'])
plt.suptitle('Generated characters grouped by the class the reference classifier assigns', y=0.93)
plt.savefig(os.path.join(OUT, 'generated_by_class.png'), dpi=150, bbox_inches='tight'); plt.show()

## 10. Automatic results summary
This section collects every number measured above into one table and writes a plain-language verdict from them,
so the conclusions always match *your* run.

In [ ]:
last = max(3, E // 5)
res = {
    'Epochs trained'                           : E,
    'Training time (minutes)'                  : round(train_minutes, 1),
    'Final D loss (mean last %d epochs)' % last: round(window('d_loss', E - last, E), 3),
    'Final G loss (mean last %d epochs)' % last: round(window('g_loss', E - last, E), 3),
    'D(x) on real (train, last epochs)'        : round(window('d_x', E - last, E), 3),
    'D(G(z)) on fake (train, last epochs)'     : round(window('d_gz', E - last, E), 3),
    'D accuracy on real TEST images'           : round(acc_real_test, 3),
    'D accuracy on generated images'           : round(acc_fake_pool, 3),
    'Reference classifier test accuracy'       : round(clf_acc, 4),
    'Classifier confidence: real / generated'  : '%.3f / %.3f' % (conf_real, conf_fake),
    'Inception-style score: real / generated'  : '%.2f / %.2f' % (is_real, is_fake),
    'Frechet distance: real-vs-real / GAN / noise': '%.1f / %.1f / %.1f' % (fd_baseline, fd_gan, fd_noise),
    'Diversity ratio (generated / real)'       : round(div_fake / div_real, 3),
    'Classes covered (>=3% each)'              : '%d / 10' % classes_covered,
}
summary = pd.DataFrame(res.items(), columns=['metric', 'value']).set_index('metric')
display(summary)
summary.to_csv(os.path.join(OUT, 'results_summary.csv'))

In [ ]:
# ---------- plain-language verdict, generated from the measured numbers
d_end, g_end = window('d_loss', E - last, E), window('g_loss', E - last, E)
a_real, a_fake = window('acc_real', E - last, E), window('acc_fake', E - last, E)
lines = []

if d_end < 0.4 and g_end > 3:
    lines.append('TRAINING BALANCE: the discriminator dominated (D loss %.2f, G loss %.2f). The generator received weak feedback; try more epochs, a smaller D, or stronger label smoothing.' % (d_end, g_end))
elif 0.9 <= d_end <= 1.6 and 0.4 <= g_end <= 2.5:
    lines.append('TRAINING BALANCE: healthy. D loss %.2f is close to the 1.386 equilibrium and G loss %.2f is in a stable range, so neither network overpowered the other.' % (d_end, g_end))
else:
    lines.append('TRAINING BALANCE: moderately uneven (D loss %.2f, G loss %.2f) but the losses stayed finite and bounded, so training did not diverge.' % (d_end, g_end))

lines.append('DISCRIMINATOR: it still labels %.0f%% of real and %.0f%% of generated images correctly during training; %.0f%% of fakes are caught on a fresh 10,000-image test.'
             % (a_real * 100, a_fake * 100, acc_fake_pool * 100))

ratio = fd_gan / max(fd_noise, 1e-9)
lines.append('IMAGE QUALITY (Frechet distance): %.1f for the GAN versus %.1f for random noise (%.0f%% of the noise value) and %.1f for real-vs-real. %s'
             % (fd_gan, fd_noise, ratio * 100, fd_baseline, 'The generated features are far closer to real data than noise is.' if ratio < 0.25 else 'There is still a clear gap to real data.'))

lines.append('CLARITY AND DIVERSITY: the classifier is %.0f%% confident on generated images (real: %.0f%%); Inception-style score %.2f of a maximum 10 (real: %.2f).'
             % (conf_fake * 100, conf_real * 100, is_fake, is_real))

if classes_covered == 10 and div_fake / div_real > 0.85:
    lines.append('MODE COLLAPSE: none detected. All 10 classes appear and pixel-level diversity is %.0f%% of the real data.' % (div_fake / div_real * 100))
elif classes_covered >= 8:
    lines.append('MODE COLLAPSE: mild. %d of 10 classes are well represented; diversity is %.0f%% of real.' % (classes_covered, div_fake / div_real * 100))
else:
    lines.append('MODE COLLAPSE: likely. Only %d of 10 classes are well represented.' % classes_covered)

lines.append('MEMORISATION: generated images are on average %.1f away from their closest training image, compared with %.1f for genuine unseen test images.'
             % (nn_dist.mean(), np.sqrt(np.clip(d2t.min(1), 0, None)).mean()))

print('\n'.join('* ' + l for l in lines))
open(os.path.join(OUT, 'verdict.txt'), 'w').write('\n'.join(lines))

In [ ]:
# zip everything produced (plots, GIF, weights, tables) and offer it for download in Colab
import shutil
archive = shutil.make_archive('kmnist_dcgan_results', 'zip', OUT)
print('Created', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print('(not running in Colab - find the zip in the working folder)')

## 11. Discussion

### 11.1 What worked
* Following the DCGAN rules (strided / transposed convolutions, batch-norm in G, LeakyReLU, tanh output, Adam 2e-4 / 0.5) gave stable training without any hyper-parameter search.
* One-sided label smoothing and dropout kept the discriminator from winning outright.
* The progression noise -> blurry blobs -> stroke-like shapes -> recognisable kuzushiji characters is visible in the progress grid and GIF.

### 11.2 Limitations
* KMNIST has 10 classes, but the GAN is **unconditional**: there is no way to ask it for a specific character. (A *conditional* GAN fixes this by feeding the class label to both networks.)
* The custom Frechet distance and Inception-style score depend on a small classifier, so they are useful for comparing runs of *this* notebook but are not standard benchmark values.
* Training was short (50 epochs); stroke ends and thin connections are still occasionally broken or blurry.
* A GAN loss is not a quality score: low loss does not mean good images, which is why image grids and the extra metrics are needed.

### 11.3 Ideas for improvement
More epochs, learning-rate decay, spectral normalisation in D, a two-time-scale update rule (different learning rates for G and D), an exponential moving average of the generator weights, a Wasserstein loss with gradient penalty (WGAN-GP), or a conditional DCGAN.

### 11.4 Conclusion
A DCGAN was implemented from scratch and trained on Kuzushiji-MNIST. The discriminator and generator losses oscillated around an
equilibrium instead of converging to zero, which is the expected behaviour of an adversarial game, and the diagnostics (D(x), D(G(z)),
accuracies, gradient norms) confirmed that neither network collapsed. The generated characters improve steadily over training; the
classifier-based confidence, Inception-style score, Frechet distance, class histogram, diversity ratio and nearest-neighbour test all
indicate realistic, varied and non-copied images. The generator therefore learned a useful approximation of the distribution of classical Japanese cursive characters.

---

## 12. Viva preparation: likely questions and short answers

| Question | Short answer |
|---|---|
| What is a GAN? | Two networks trained against each other: a generator that makes fake data from noise and a discriminator that separates real from fake. |
| Why is there no label used in training? | A plain GAN is unsupervised; it only needs to know real vs generated, not the class. |
| Why scale images to [-1, 1]? | The generator ends with `tanh`, whose output is in [-1, 1]; both real and fake must share a range. |
| Why LeakyReLU instead of ReLU in D? | It lets a small gradient pass for negative inputs, so units do not die and G keeps receiving gradients. |
| Why strided convolutions rather than max-pooling? | They let the network learn its own downsampling. |
| Why BatchNorm in the generator? | It stabilises training and helps gradients flow in deep generators. |
| What is the ideal discriminator loss? | `2 ln 2 = 1.386`, meaning D outputs 0.5 for everything. |
| Why do the losses not go to zero? | Lowering one player's loss raises the other's; it is a zero-sum game with an equilibrium, not an optimisation to zero. |
| What is mode collapse? | The generator produces only a few distinct outputs because they reliably fool D. |
| What is label smoothing? | Using 0.9 instead of 1.0 as the "real" target so D does not become over-confident. |
| Why use two optimisers and tapes? | Each network must be updated only by its own loss. |
| What does `training=True` do? | Turns on dropout in D and uses batch statistics in BatchNorm of G. |
| How is image quality measured? | By eye plus metrics: classifier confidence, Inception score, Frechet distance, class coverage, diversity. |
| What is the latent space? | The 100-dimensional space of noise vectors; nearby points give similar images, so interpolation gives smooth morphs. |

---

## 13. References
1. I. Goodfellow et al., *Generative Adversarial Nets*, NeurIPS 2014.
2. A. Radford, L. Metz, S. Chintala, *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks*, ICLR 2016.
3. T. Clanuwat et al., *Deep Learning for Classical Japanese Literature*, 2018 (the KMNIST dataset).
4. T. Salimans et al., *Improved Techniques for Training GANs*, NeurIPS 2016 (Inception score, label smoothing).
5. M. Heusel et al., *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium*, NeurIPS 2017 (FID).
6. TensorFlow / Keras documentation, *Customizing what happens in train_step* and *DCGAN tutorial*.